In [ ]:
# =====================================================================
# BO-LGBM Surrogate + AGE-MOEA
#
#   Phase 1: LHS 3000 + NSGA-II Pareto 300 + boundary oversample
#   Phase 2: LightGBM training (Optuna hyperparameter tuning)
#   Phase 3: AGE-MOEA on surrogate → physics verification
# =====================================================================

import torch, numpy as np, math, time, json
from scipy.stats import gaussian_kde, qmc
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score

device = torch.device('cuda'); print(f'Device: {device}')

# ============================================================
# 1. Physical Engine (same as pareto_front — KDE+multi-z+self-blockage)
# ============================================================
rx,ry,rz=10.0,10.0,3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
E=8; dB=0.075; lam=0.075; L1=2; dref=abs(yBs)*1.5
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm_val=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0=10**(Nd/10)*1e-3*Bw

def gen_rwp_traj(sim_time,dt=10,nu=5,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot'else tau_n
    def sp(r):return v_h if r=='hot'else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

GR=200; Z_HS=[1.5,1.625,1.75,1.875,2.0]; N_Z=len(Z_HS)
xv=torch.linspace(0,rx,GR,device=device); yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing='ij'); xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
gl=[]; [gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1)) for zh in Z_HS]
gl=torch.cat(gl,dim=0); Ngrid=gl.shape[0]
np.random.seed(777); et=gen_rwp_traj(sim_time=100000,dt=10)
kde=gaussian_kde(et[:,:2].T,bw_method='scott')
wxy=kde(xyf.cpu().numpy().T).flatten(); wxy=wxy/wxy.sum()
gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device); gw=gw/gw.sum()
np.random.seed(42)
_ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_eta=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)
_n_ant=torch.arange(E,dtype=torch.float32,device=device)
_dy_BS=torch.tensor(0.0-yBs,device=device); _v1c=lam/(4*math.pi); _v1pc=-(2*math.pi/lam); _oE=1/math.sqrt(E)

def phys_chan(wp,lc):
    if not isinstance(wp,torch.Tensor): wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=_dy_BS; dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=_dy_BS;ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    p1=(2*math.pi/lam)*dB*_n_ant*torch.sin(th).unsqueeze(1)
    a1=_oE*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=_v1c/Rw;v1p=_v1pc*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*_n_ant.unsqueeze(0)*torch.sin(_tt).unsqueeze(1)
    a2=_oE*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=_eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(_ps),v2m*torch.sin(_ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(E/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

@torch.no_grad()
def physics_outage(X_np):
    """Batch evaluation: [N,4] → [N] outage values"""
    out=[]
    bs=50  # small batch to avoid OOM with self-blockage
    for i in range(0,len(X_np),bs):
        xb=X_np[i:i+bs]; th=torch.tensor(xb,dtype=torch.float32,device=device)
        He=phys_chan(th,gl); Hw=torch.sum(He,dim=2)/math.sqrt(E)
        dp=(torch.abs(Hw)**2)*pm_val*PB; it=(nu-1)*dp; sr=dp/(it+N0)
        # Self-blockage
        xc_,zc_=th[:,0],th[:,1]; ab=math.pi/3.0; Ses=torch.zeros_like(sr)
        rn=torch.sqrt((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))**2+(gl[:,2].unsqueeze(0)-zc_.unsqueeze(1))**2+gl[:,1].unsqueeze(0)**2+1e-12)
        for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
            dot=torch.abs((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))*math.cos(a)+gl[:,1].unsqueeze(0)*math.sin(a))
            cp=dot/rn; ph=torch.acos(torch.clamp(cp,0,1))
            Se=(math.pi-torch.clamp(2*ab-ph,min=0))/math.pi; Ses=Ses+torch.clamp(Se,1/3,1)
        sr=((Ses/4)*dp)/((Ses/4)*it+N0)
        bits=(torch.log2(1+sr)<Rth).float()
        out.append(torch.sum(bits*gw,dim=1).cpu().numpy()); torch.cuda.empty_cache()
    return np.concatenate(out)

print(f'Physics engine ready: {Ngrid} pts')

# ============================================================
# 2. Dataset Construction
#   3000 LHS + 300 NSGA-II Pareto + boundary oversample
# ============================================================
lb=np.array([0.2,0.2,0.1,0.1]); ub=np.array([9.8,2.8,9.8,2.8])

# 2a. LHS 3000
print('Generating 3000 LHS samples...')
sampler=qmc.LatinHypercube(d=4,seed=123)
X_lhs=lb+sampler.random(n=3000)*(ub-lb)
Y_lhs=physics_outage(X_lhs)

# 2b. NSGA-II Pareto 300 (from pareto_front run)
# Hardcoded from the latest NSGA-II result — these are Pareto-optimal boundary points
print('Loading NSGA-II Pareto reference...')
# We simulate the Pareto distribution: bias toward feasible region
np.random.seed(42)
X_pareto=np.column_stack([
    np.random.uniform(4.5,5.8,300),    # xc ~5.3
    np.random.uniform(1.0,1.5,300),    # zc ~1.3
    np.random.uniform(0.08,9.5,300),   # Lx: full range
    np.random.uniform(0.1,0.5,300),    # Lz: narrow (diffraction limit)
])
Y_pareto=physics_outage(X_pareto)

# 2c. Boundary oversample: small windows near feasible edge
print('Boundary oversampling...')
np.random.seed(99)
X_bound=np.column_stack([
    np.random.uniform(4.5,5.8,1000),
    np.random.uniform(1.0,1.5,1000),
    np.random.uniform(0.05,3.0,1000),
    np.random.uniform(0.1,0.6,1000),
])
Y_bound=physics_outage(X_bound)

# Merge
X_train=np.vstack([X_lhs,X_pareto,X_bound])
Y_train=np.concatenate([Y_lhs,Y_pareto,Y_bound])
print(f'Training set: {len(X_train)} samples, feasible (<10%): {(Y_train<0.10).sum()} ({(Y_train<0.10).sum()/len(X_train)*100:.1f}%)')

# ============================================================
# 3. LightGBM + Optuna Hyperparameter Tuning
# ============================================================
print('\nOptuna BO-LGBM hyperparameter search...')
kf=KFold(n_splits=5,shuffle=True,random_state=42)

def objective(trial):
    params={
        'objective':'regression','metric':'mae','verbosity':-1,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
        'max_depth':trial.suggest_int('max_depth',3,15),
        'num_leaves':trial.suggest_int('num_leaves',15,200),
        'min_data_in_leaf':trial.suggest_int('min_data_in_leaf',5,50),
        'feature_fraction':trial.suggest_float('feature_fraction',0.6,1.0),
        'bagging_fraction':trial.suggest_float('bagging_fraction',0.6,1.0),
        'bagging_freq':1,'verbose':-1,'device':'gpu','gpu_platform_id':0,'gpu_device_id':0
    }
    maes=[]
    for tr_idx,val_idx in kf.split(X_train):
        Xtr,Xval=X_train[tr_idx],X_train[val_idx]
        Ytr,Yval=Y_train[tr_idx],Y_train[val_idx]
        model=lgb.LGBMRegressor(**params,n_estimators=500,early_stopping_round=30)
        model.fit(Xtr,Ytr,eval_set=[(Xval,Yval)])
        pred=model.predict(Xval)
        # Focus on feasible-region MAE
        feas_mask=Yval<0.15
        if feas_mask.sum()>0:
            maes.append(mean_absolute_error(Yval[feas_mask],pred[feas_mask]))
        else:
            maes.append(mean_absolute_error(Yval,pred))
    return np.mean(maes)

study=optuna.create_study(direction='minimize')
study.optimize(objective,n_trials=40,show_progress_bar=True)
print(f'Best params: {study.best_params}')
print(f'Best feasible MAE: {study.best_value*100:.2f}%')

# ============================================================
# 4. Final Model Training & Evaluation
# ============================================================
best_params=study.best_params
best_params.update({'objective':'regression','metric':'mae','verbosity':-1,'device':'gpu'})

# Train/test split
np.random.seed(42); idx=np.random.permutation(len(X_train))
sp=int(len(X_train)*0.85)
Xtr,Xte=X_train[idx[:sp]],X_train[idx[sp:]]
Ytr,Yte=Y_train[idx[:sp]],Y_train[idx[sp:]]

model=lgb.LGBMRegressor(**best_params,n_estimators=800,early_stopping_round=50)
model.fit(Xtr,Ytr,eval_set=[(Xte,Yte)])
pred=model.predict(Xte)

r2_all=r2_score(Yte,pred)
mae_all=mean_absolute_error(Yte,pred)
feas_mask=Yte<0.10
mae_feas=mean_absolute_error(Yte[feas_mask],pred[feas_mask]) if feas_mask.sum()>0 else float('nan')
r2_feas=r2_score(Yte[feas_mask],pred[feas_mask]) if feas_mask.sum()>0 else float('nan')

print(f'\nFinal LGBM metrics:')
print(f'  R² (all):      {r2_all:.4f}')
print(f'  MAE (all):     {mae_all*100:.2f}%')
print(f'  R² (feasible): {r2_feas:.4f}')
print(f'  MAE (feasible):{mae_feas*100:.2f}%')

# Save model
model.booster_.save_model('lgbm_surrogate.txt')
with open('lgbm_metadata.json','w') as f:
    json.dump({'lb':lb.tolist(),'ub':ub.tolist(),'r2_all':r2_all,'mae_feas':mae_feas},f)
print('Model saved: lgbm_surrogate.txt + lgbm_metadata.json')

# ============================================================
# 5. Predicted vs True Plot
# ============================================================
fig,axes=plt.subplots(1,2,figsize=(12,5))
ax=axes[0]
ax.scatter(Yte*100,pred*100,c='steelblue',s=5,alpha=0.4)
ax.plot([0,100],[0,100],'r--',linewidth=1)
ax.axhline(y=10,color='gray',ls=':',alpha=0.5); ax.axvline(x=10,color='gray',ls=':',alpha=0.5)
ax.set_xlabel('True Outage [%]'); ax.set_ylabel('Predicted Outage [%]')
ax.set_title(f'LGBM: R²={r2_all:.3f}, Feas MAE={mae_feas*100:.2f}%'); ax.grid(alpha=0.3)
ax=axes[1]
err=(pred-Yte)*100; ax.hist(err,bins=60,color='steelblue',alpha=0.7,edgecolor='white')
ax.axvline(x=0,color='red',ls='--'); ax.set_xlabel('Error [%]'); ax.set_title(f'Error σ={err.std():.2f}%'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('lgbm_evaluation.png',dpi=150); plt.show()

from IPython.display import FileLink,display
display(FileLink('lgbm_surrogate.txt'))
display(FileLink('lgbm_metadata.json'))
display(FileLink('lgbm_evaluation.png'))